# Test the Extraction Service

Quick smoke test for the extraction API — confirms it works for **both a PDF
and an image**, using two synthetic sample documents included in this repo
(`sample_docs/`). No real customer documents needed.

**Before you start:**
1. Run the API locally (from the repo root): `uvicorn app.main:app --reload`
2. Have a Gemini API key ready: https://aistudio.google.com/app/apikey
3. `pip install -r requirements-dev.txt` (adds `requests` used by this notebook)


In [ ]:
import requests

BASE_URL = "http://localhost:8000"   # change if testing a deployed Cloud Run URL
GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"   # or: import os; os.environ["GEMINI_API_KEY"]


## 1. Health check

In [ ]:
r = requests.get(f"{BASE_URL}/health")
print(r.status_code, r.json())
assert r.ok


## 2. List available templates
Confirms the service loaded `templates_config.json` correctly.

In [ ]:
r = requests.get(f"{BASE_URL}/templates")
templates = r.json()
for t in templates:
    print(f"{t['key']:35s} kind={t['kind']:6s} fields={t['field_count']}")


## 3. Generate the sample documents (if you haven't already)
Creates `sample_docs/sample_ssm_certificate.pdf` and `sample_docs/sample_ic_photocopy.png`.
Safe to re-run.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "make_sample_docs.py"], cwd=".", check=True)


## 4. Test PDF extraction
Template: `business_registration_ssm` — a single-object template (`kind: single`).


In [ ]:
with open("../sample_docs/sample_ssm_certificate.pdf", "rb") as f:
    r = requests.post(
        f"{BASE_URL}/extract",
        data={"template": "business_registration_ssm", "api_key": GEMINI_API_KEY},
        files={"file": ("sample_ssm_certificate.pdf", f, "application/pdf")},
        timeout=60,
    )

print(r.status_code)
result_pdf = r.json()
result_pdf


In [ ]:
# Quick sanity check on the extracted fields
assert r.status_code == 200, result_pdf
extracted = result_pdf["data"]["extracted_data"]
print("business_name:", extracted.get("business_name"))
print("business_registration_number:", extracted.get("business_registration_number"))
print("incorporation_date:", extracted.get("incorporation_date"))


## 5. Test image extraction
Template: `ic_photocopies` — an array-of-objects template (`kind: array`), since a
document set can contain multiple ICs (one per director/owner).


In [ ]:
with open("../sample_docs/sample_ic_photocopy.png", "rb") as f:
    r = requests.post(
        f"{BASE_URL}/extract",
        data={"template": "ic_photocopies", "api_key": GEMINI_API_KEY},
        files={"file": ("sample_ic_photocopy.png", f, "image/png")},
        timeout=60,
    )

print(r.status_code)
result_img = r.json()
result_img


In [ ]:
assert r.status_code == 200, result_img
extracted = result_img["data"]["extracted_data"]
assert isinstance(extracted, list), "ic_photocopies is an array-kind template"
print(f"Extracted {len(extracted)} individual(s):")
for person in extracted:
    print(" -", person.get("name"), "|", person.get("nric"))


## 6. Error handling checks (optional)
Confirms bad input is rejected cleanly rather than silently mishandled.


In [ ]:
# Wrong template name -> 404
r = requests.post(
    f"{BASE_URL}/extract",
    data={"template": "does_not_exist", "api_key": GEMINI_API_KEY},
    files={"file": ("x.pdf", open("../sample_docs/sample_ssm_certificate.pdf", "rb"), "application/pdf")},
)
print("unknown template ->", r.status_code, r.json())
assert r.status_code == 404


In [ ]:
# Unsupported file type -> 400
r = requests.post(
    f"{BASE_URL}/extract",
    data={"template": "consent_form", "api_key": GEMINI_API_KEY},
    files={"file": ("x.txt", b"hello world", "text/plain")},
)
print("unsupported mime ->", r.status_code, r.json())
assert r.status_code == 400


## Done
If both extraction cells above returned `200` with plausible field values,
the service works end to end for both file types. Next: point `BASE_URL` at
your deployed Cloud Run URL and re-run cells 1–5 to confirm production works too.
